# Import libraries

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("wandb")
!wandb login $secret_value

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
pd.set_option('max_columns', None)
pd.set_option('max_rows', None)

import wandb
from wandb.keras import WandbCallback

from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import Concatenate, LSTM, GRU, Masking
from tensorflow.keras.layers import Bidirectional, Multiply
from tensorflow.keras import backend as K

# Set the random seeds
os.environ['TF_CUDNN_DETERMINISTIC'] = '1' 
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(42)
tf.random.set_seed(42)

# Custom function

In [ ]:
def plot_training(history):
    # list all data in history
    print(history.history.keys())
    # summarize history for loss
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()

In [ ]:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

# Load source datasets

In [ ]:
train_df = pd.read_csv('../input/ventilator-pressure-prediction/train.csv')
print(f"train_df: {train_df.shape}")
train_df.head()

In [ ]:
test_df = pd.read_csv('../input/ventilator-pressure-prediction/test.csv')
print(f"test_df: {test_df.shape}")
test_df.head()

In [ ]:
targets = train_df[['pressure']].to_numpy().reshape(-1, 80)
print(targets.shape)

del train_df
del test_df
_ = gc.collect()

# Feature Engineering

In [ ]:
# Read in dataset with features created
train = pd.read_feather("../input/gbvpp-featureset102/vast_train.feather")
test = pd.read_feather("../input/gbvpp-featureset102/vast_test.feather")

print(train.shape)
print(test.shape)

In [ ]:
dropCols = ['u_in_lag_back1','u_out_lag_back1','u_in_lag2','u_out_lag2','u_in_lag_back2','u_out_lag_back2',
'u_in_lag3','u_out_lag3','u_in_lag_back3','u_out_lag_back3','u_in_lag4','u_out_lag4','u_in_lag_back4',
'u_out_lag_back4','breath_id__u_in__max','breath_id__u_in__mean']

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

print(f"train: {train.shape} \ntest: {test.shape}")

In [ ]:
# Drop columns based on permutation importance
dropCols = ['breath_id__u_in__diffmax','ewm_u_in_corr','u_in_grp_R__C_steps_max_diff','u_out_diff1',
'u_out_lagback_diff2','u_out_lagback_diff1','u_out_lag3','u_out_lag2','unwrapped_phase',
'u_in_grp_R__C_steps_max','phase_shift1','u_in_grp_R__C_steps_min','u_out_lag1','u_out_lag4',
'u_out_lag_back2','u_out_lag_back1','u_out_lag_back4','u_out_lag_back3','cross2',
'time_step_cumsum','plus_one','minus_one','time_step','u_in_first','breath_id__u_in__diffmean',
'ewm_u_in_std','phase','area','expand_max','breath_id__u_in__max','expand_std']
dropCols = [col for col in dropCols if col in train.columns]

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

COLS = train.columns.tolist()

print(train.shape)
print(test.shape)

In [ ]:
# Drop columns based on permutation importance
dropCols = ['time_passed','time_passed_grp_R__C_steps_mean','time_passed_grp_R__C_steps_min','u_out_diff2','u_out_diff4',
'u_in_rolling_mean_lead15','u_in_rolling_max_lead15','u_in_rolling_min_lead15','u_in_rolling_std_lead15',
'fft_u_in','fft_u_in_w','envelope','IF','time_gap']
dropCols = [col for col in dropCols if col in train.columns]

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

COLS = train.columns.tolist()

print(train.shape)
print(test.shape)

In [ ]:
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

In [ ]:
uout_train = train['u_out'].values
uout_train = uout_train.reshape(-1, 80)
uout_train = np.expand_dims(uout_train, axis=-1)
print(uout_train.shape)

uout_test = test['u_out'].values
uout_test = uout_test.reshape(-1, 80)
uout_test = np.expand_dims(uout_test, axis=-1)
print(uout_test.shape)

In [ ]:
quantVal = 15

scaler = RobustScaler(quantile_range=(quantVal, 100-quantVal))
train = scaler.fit_transform(train)
test = scaler.transform(test)

train = train.reshape(-1, 80, train.shape[-1])
train_inhale = train*(1-uout_train)
# train_exhale = train*uout_train

test = test.reshape(-1, 80, train.shape[-1])
test_inhale = test*(1-uout_test)
# test_exhale = test*uout_test

# print(f"train: {train.shape} \ntrain_inhale: {train_inhale.shape} \ntrain_exhale: {train_exhale.shape}")
print(f"train: {train.shape} \ntrain_inhale: {train_inhale.shape}")
print(f"test: {test.shape} \ntest_inhale: {test_inhale.shape}")
print(f"\n\ntargets: {targets.shape}")

In [ ]:
pressure = targets.squeeze().reshape(-1,1).astype('float32')

P_MIN = np.min(pressure)
P_MAX = np.max(pressure)
P_STEP = (pressure[1] - pressure[0])[0]
print('Min pressure: {}'.format(P_MIN))
print('Max pressure: {}'.format(P_MAX))
print('Pressure step: {}'.format(P_STEP))
print('Unique values:  {}'.format(np.unique(pressure).shape[0]))

del pressure
_ = gc.collect()

# Hardware config

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
    BATCH_SIZE = strategy.num_replicas_in_sync * 64
    print("Running on TPU:", tpu.master())
    print(f"Batch Size: {BATCH_SIZE}")
    
except ValueError:
    strategy = tf.distribute.get_strategy()
    BATCH_SIZE = 256
    print(f"Running on {strategy.num_replicas_in_sync} replicas")
    print(f"Batch Size: {BATCH_SIZE}")

# Cerberus Model

In [ ]:
def CerberusModel():
    
    input_A = Input(shape=(train.shape[-2:]))
    input_B = Input(shape=(train_inhale.shape[-2:]))
#     input_C = Input(shape=(train_exhale.shape[-2:]))
    
    layers = [400,200,100]
    
    x_A = Bidirectional(LSTM(units=layers[0], return_sequences=True))(input_A)
    x_A = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x_A)
    x_A = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x_A)
    
    input_B_masked = Masking(mask_value=0., input_shape=(train_inhale.shape[-2:]))(input_B)
    x_B = Bidirectional(LSTM(units=layers[0], return_sequences=True))(input_B_masked)
    x_B = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x_B)
    x_B = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x_B)
    
#     input_C_masked = Masking(mask_value=0., input_shape=(train_exhale.shape[-2:]))(input_C)
#     x_C = Bidirectional(LSTM(units=layers[0], return_sequences=True))(input_C_masked)
#     x_C = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x_C)
#     x_C = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x_C)
    
    x = keras.layers.Dense(1000, activation='relu')(tf.concat([x_A, x_B], axis=-1))

    y_hat = keras.layers.Dense(1)(x)
    
    model = keras.models.Model(inputs=[input_A, input_B], outputs=y_hat)
    
    
#     x = Concatenate(axis=2)([x_A, x_B, x_C])
    
#     layers = [200,100]
    
#     x = Bidirectional(GRU(units=layers[0], return_sequences=True))(x)
    
# #     x = Multiply()([x_A1, x])
# #     x = BatchNormalization()(x)
# #     x = Bidirectional(GRU(units=layers[1], return_sequences=True))(x)
    
# #     x = Concatenate(axis=2)([x_A, x])
    
#     x1 = Dense(units=100, activation='selu')(x)
#     x2 = Dense(units=100, activation='selu')(x1)
# #     x3 = Concatenate(axis=2)([x2,x1])
#     x3 = Dense(units=50, activation='selu')(x2)
    
#     x_output = Dense(units=1)(x3)

#     model = Model(inputs=[input_A, input_B, input_C], outputs=x_output, 
#                   name='Cerberus')
    return model

In [ ]:
model = CerberusModel()
model.summary()

In [ ]:
plot_model(
    model, 
    to_file='Cerberus_Model.png', 
    show_shapes=True,
    show_layer_names=True
)

# Model training

In [ ]:
VERBOSE = 1
ONE_FOLD_ONLY = True 
COMPUTE_LSTM_IMPORTANCE = False

FOLDS = 7
SEED = 101

# Learning rate
INIT_LR = 0.001

# ReduceLROnPlateau
FACTOR = 0.8
LR_PATIENCE = 4
MIN_LR = 5e-6
LR_DELTA = 0

# Early Stopping
ES_DELTA = 0
PATIENCE = 25

# Fit
EPOCHS = 300

In [ ]:
with strategy.scope():

    test_preds = []
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(kf.split(train, targets)):


        print(f"Fold {fold+1}")
        X_train, X_valid = train[train_idx], train[test_idx]
        X_train_inhale, X_valid_inhale = train_inhale[train_idx], train_inhale[test_idx]
    #     X_train_exhale, X_valid_exhale = train_exhale[train_idx], train_exhale[test_idx]

        y_train, y_valid = targets[train_idx], targets[test_idx]

        if not ONE_FOLD_ONLY:

            print("Init wandb logs")

            # Initialize wandb with your project name
            run = wandb.init(project='GBVPP',
                             config={  # and include hyperparameters and metadata
                                 "FOLDS": FOLDS,
                                 "SEED": SEED,
                                 "ROBUSTSCALER QUANT": quantVal,
                                 "INIT_LR": INIT_LR,
                                 "RLROP FACTOR": FACTOR,
                                 "RLROP LR_PATIENCE": LR_PATIENCE,
                                 "RLROP MIN_LR": MIN_LR,
                                 "RLROP DELTA": LR_DELTA,
                                 "ES DELTA": ES_DELTA,
                                 "PATIENCE": PATIENCE,
                                 "EPOCHS": EPOCHS,
                                 "BATCH_SIZE": BATCH_SIZE,
                                 "LOSS": "mae",
                                 "MODEL": "CerberusModel", 

                             })
            config = wandb.config

            # Wandb
            wnb = WandbCallback()

        print("Model init")

        model = CerberusModel()
        model.compile(optimizer="adam", loss="mae") #, sample_weight_mode="temporal"

        # ModelCheckpoint
        save_locally = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')
        chk_point = ModelCheckpoint(f'./CerberusModel_{SEED}_fold{fold+1}.h5', options=save_locally, 
                                    monitor='val_loss', verbose=VERBOSE, 
                                    save_best_only=True, mode='min')

        # Learning Rate Schedule
        lr = ReduceLROnPlateau(monitor="val_loss", factor=FACTOR, 
                               patience=LR_PATIENCE, verbose=VERBOSE,
                               min_lr=MIN_LR, min_delta=LR_DELTA)

        # Early Stopping Criterion
        es = EarlyStopping(monitor="val_loss", patience=PATIENCE, 
                           verbose=VERBOSE, mode="min", 
                           restore_best_weights=True,
                           min_delta=ES_DELTA)



        # Set Learning Rate
        K.set_value(model.optimizer.learning_rate, INIT_LR)
        print("Initial Learning rate:", model.optimizer.learning_rate.numpy())

        # Callbacks
        if ONE_FOLD_ONLY:
            callbacks = [lr, chk_point, es]
        else:
            callbacks = [lr, chk_point, es, wnb]

        # Model training
        print("Model training begins")
        history = model.fit([X_train, X_train_inhale], y_train, 
                          validation_data=([X_valid, X_valid_inhale], y_valid), 
                          epochs=EPOCHS,
                          verbose=VERBOSE,
                          batch_size=BATCH_SIZE, 
                          callbacks=callbacks)

        plot_training(history)

        if COMPUTE_LSTM_IMPORTANCE:
            results = []
            print(' Computing LSTM feature importance...')

            # COMPUTE BASELINE (NO SHUFFLE)
            oof_preds = model.predict([X_valid, X_valid_inhale], verbose=0).squeeze() 
            baseline_mae = np.mean(np.abs( oof_preds-y_valid ))
            results.append({'feature':'BASELINE','mae':baseline_mae})           

            for k in tqdm(range(len(COLS))):

                # SHUFFLE FEATURE K
                save_col = X_valid[:,:,k].copy()
                np.random.shuffle(X_valid[:,:,k])

                # COMPUTE OOF MAE WITH FEATURE K SHUFFLED
                oof_preds = model.predict(X_valid, verbose=0).squeeze() 
                mae = np.mean(np.abs( oof_preds-y_valid ))
                results.append({'feature':COLS[k],'mae':mae})
                X_valid[:,:,k] = save_col

            # DISPLAY LSTM FEATURE IMPORTANCE
            print()
            df = pd.DataFrame(results)
            df = df.sort_values('mae')
            plt.figure(figsize=(10,20))
            plt.barh(np.arange(len(COLS)+1),df.mae)
            plt.yticks(np.arange(len(COLS)+1),df.feature.values)
            plt.title('LSTM Feature Importance',size=16)
            plt.ylim((-1,len(COLS)+1))
            plt.plot([baseline_mae,baseline_mae],[-1,len(COLS)+1], '--', color='orange',
                     label=f'Baseline OOF\nMAE={baseline_mae:.3f}')
            plt.xlabel(f'Fold {fold+1} OOF MAE with feature permuted',size=14)
            plt.ylabel('Feature',size=14)
            plt.legend()
            plt.show()

            # SAVE LSTM FEATURE IMPORTANCE
            df = df.sort_values('mae',ascending=False)
            df.to_csv(f'lstm_feature_importance_fold_{fold+1}.csv',index=False)

        # ONLY DO ONE FOLD
        if ONE_FOLD_ONLY: break

        ######################################################



        load_locally = tf.saved_model.LoadOptions(experimental_io_device='/job:localhost')
        model = load_model(f'./CerberusModel_{SEED}_fold{fold+1}.h5', options=load_locally)

        print("Calculate OOF score")
        y_true = y_valid.squeeze().reshape(-1, 1)
        y_pred = model.predict([X_valid, X_valid_inhale], batch_size=BATCH_SIZE).squeeze().reshape(-1, 1)
        score = mean_absolute_error(y_true, y_pred)
        print(f"Fold-{fold+1} | OOF Score: {score}")

        test_preds.append(model.predict([test, test_inhale], batch_size=BATCH_SIZE).squeeze().reshape(-1, 1).squeeze())


    if not COMPUTE_LSTM_IMPORTANCE: wandb.finish()

In [ ]:
# with tpu_strategy.scope():
    
#     VERBOSE = 1
#     test_preds = []
#     kf = KFold(n_splits=10, shuffle=True, random_state=2021)
    
#     for fold, (train_idx, test_idx) in enumerate(kf.split(train, targets)):
        
#         print(f"Fold {fold+1}")
#         X_train, X_valid = train[train_idx], train[test_idx]
#         y_train, y_valid = targets[train_idx], targets[test_idx]
        
#         print("Model init")
        
#         model = dnn_model()
#         model.compile(optimizer="adam", loss="mae")

#         lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, 
#                                patience=4, verbose=VERBOSE)
        
#         save_locally = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')
#         chk_point = ModelCheckpoint(f'./Cerberus_model_2021_{fold+1}C.h5', options=save_locally, 
#                                     monitor='val_loss', verbose=VERBOSE, 
#                                     save_best_only=True, mode='min')

#         es = EarlyStopping(monitor="val_loss", patience=15, 
#                            verbose=VERBOSE, mode="min", 
#                            restore_best_weights=True)
        
#         print("Model training begins")
#         model.fit(X_train, y_train, 
#                   validation_data=(X_valid, y_valid), 
#                   epochs=250,
#                   verbose=VERBOSE,
#                   batch_size=BATCH_SIZE, 
#                   callbacks=[lr, chk_point, es])
        
#         load_locally = tf.saved_model.LoadOptions(experimental_io_device='/job:localhost')
#         model = load_model(f'./Bidirect_LSTM_model_2021_{fold+1}C.h5', options=load_locally)
        
#         print("Calculate OOF score")
#         y_true = y_valid.squeeze().reshape(-1, 1)
#         y_pred = model.predict(X_valid, batch_size=BATCH_SIZE).squeeze().reshape(-1, 1)
#         score = mean_absolute_error(y_true, y_pred)
#         print(f"Fold-{fold+1} | OOF Score: {score}")
        
#         test_preds.append(model.predict(test, batch_size=BATCH_SIZE).squeeze().reshape(-1, 1).squeeze())

# Create submission file

In [ ]:
# !ls

In [ ]:
# load_locally = tf.saved_model.LoadOptions(experimental_io_device='/job:localhost')
# model = load_model(f'./Bidirect_LSTM_model_2021_1C.h5', options=load_locally)


# test_preds = model.predict(test, batch_size=BATCH_SIZE).squeeze().reshape(-1, 1).squeeze()
# test_preds.shape

In [ ]:
# submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
# submission["pressure"] = test_preds
# submission.to_csv('fold1_submission.csv', index=False)

In [ ]:
submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
submission["pressure"] = sum(test_preds)/5
submission.to_csv('mean_submission.csv', index=False)

In [ ]:
submission["pressure"] = np.median(np.vstack(test_preds),axis=0)
submission["pressure"] = np.round((submission.pressure - P_MIN)/P_STEP) * P_STEP + P_MIN
submission["pressure"] = np.clip(submission.pressure, P_MIN, P_MAX)
submission.to_csv('median_submission.csv', index=False)